[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/AD7031/blob/main/notebooks/week06_markowitz_max_return.ipynb)

In [ ]:
%pip install -q gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB

params = {
    "WLSACCESSID": "여기에 ACCESSID 붙여넣기",
    "WLSSECRET":   "여기에 SECRET 붙여넣기",
    "LICENSEID":   123456,   # 숫자 그대로, 따옴표 없이
}
env = gp.Env(params=params)

# Markowitz, flipped — maximize the expected return

$$
\begin{aligned}
\max_{\mathbf{x}} \quad & \mathbb{E}\big[\tilde\xi^\top \mathbf{x}\big] = \mu^\top \mathbf{x} \\
\text{s.t.} \quad & e^\top \mathbf{x} \le 1 \\
                  & \mathbf{x}^\top \Sigma\, \mathbf{x} \le 0.13^2 \\
                  & \mathbf{x} \ge 0
\end{aligned}
$$

Maximize the **expected return**, with the **risk capped**: $\sigma \le 13\%$. $\tilde\xi$ is the vector of asset returns, $e$ is the all-ones vector.

Two assets, so the whole problem fits on one 2-D picture — $\mathbf{x} = (x, y)^\top$, the shares of the budget in each:

| asset | share | expected return $\mu_j$ | variance $\Sigma_{jj}$ | $\sigma_j$ |
| :--- | :---: | ---: | ---: | ---: |
| asset 1 | $x$ | 10% | 0.01 | 10% |
| asset 2 | $y$ | 15% | 0.04 | 20% |

Covariance: $\Sigma_{12} = 0.001$.

$$
\mu = \begin{pmatrix} 0.10 \\ 0.15 \end{pmatrix}, \qquad
\Sigma = \begin{pmatrix} 0.01 & 0.001 \\ 0.001 & 0.04 \end{pmatrix}, \qquad
e = \begin{pmatrix} 1 \\ 1 \end{pmatrix}
$$

Written out:

$$
\begin{aligned}
\max_{x,\,y} \quad & 0.10\,x + 0.15\,y \\
\text{s.t.} \quad & x + y \le 1 \\
                  & 0.01x^{2} + 0.04y^{2} + 0.002xy \le 0.0169 \\
                  & x,\ y \ge 0
\end{aligned}
$$

## Live coding — fill in the blanks

Skeleton for building the model live in class, one comment at a time.

In [ ]:
pf = gp.Model("max_return_risk_cap")

# decision variables, two continuous shares x and y --- pf.addVar(vtype=GRB.CONTINUOUS)


# objective: expected return, and we MAXIMIZE this time --- pf.setObjective(..., GRB.MAXIMIZE)


# constraints: variance <= 0.13**2 -- quadratic, in a constraint this time! -- and the budget --- pf.addConstr()


# model is written down -- let's read it back before solving --- pf.update(), pf.write("max_return.lp"), print(GREEN + open("max_return.lp").read() + RESET)
# --- e.g., GREEN = "\033[92m"; RESET = "\033[0m"

In [ ]:
# solve --- pf.optimize()


# print x, y, the expected return, and the risk sigma = sqrt(x' Sigma x) -- in GREEN

## Reference: the completed model

By hand — no arrays, no loops, no `quicksum` — so every term matches the math above.

In [ ]:
pf = gp.Model("max_return_risk_cap")

# decision variables -- one share per asset, continuous, lower bound 0 by default
x = pf.addVar(vtype=GRB.CONTINUOUS, name="x")
y = pf.addVar(vtype=GRB.CONTINUOUS, name="y")

# objective -- expected return
pf.setObjective(0.10 * x + 0.15 * y, GRB.MAXIMIZE)

# constraints -- portfolio variance at most 0.13^2 (sigma at most 13%); invest at most the whole fund
sigma_max = 0.13
pf.addConstr(0.01 * x * x + 0.04 * y * y + 0.002 * x * y <= sigma_max ** 2, name="risk_cap")
pf.addConstr(x + y <= 1, name="budget")

# read the model back before solving -- the quadratic shows up inside a constraint, under [ ... ]
GREEN, RESET = "\033[92m", "\033[0m"          # the green from the first colab session
pf.update()
pf.write("max_return.lp")
print(GREEN + open("max_return.lp").read() + RESET)

## Same model, vector form — `mu.T @ x`

gurobipy also has a **matrix API**: `addMVar` creates the whole vector $\mathbf{x}$ at once, and `@` is matrix multiplication — so the code reads like the canonical form at the top.

| math | code |
| :--- | :--- |
| $\mu^\top \mathbf{x}$ | `mu.T @ x` |
| $\mathbf{x}^\top \Sigma\, \mathbf{x} \le 0.13^2$ | `x.T @ Sigma @ x <= 0.13 ** 2` |
| $e^\top \mathbf{x} \le 1$ | `e.T @ x <= 1` |
| $\mathbf{x} \ge 0$ | `lb=0.0` in `addMVar` |

Name the two entries `x` and `y`, and Gurobi prints **exactly the same model** as the by-hand cell right above — the same formulation, encoded two ways. With 50 assets only `mu`, `Sigma`, and `e` grow — the model lines stay exactly as they are.

In [ ]:
import numpy as np

mu    = np.array([0.10, 0.15])          # expected returns
Sigma = np.array([[0.01,  0.001],
                  [0.001, 0.04 ]])      # covariance matrix
e     = np.ones(2)                      # all-ones vector

pv = gp.Model("max_return_risk_cap")

# decision variables -- one MVar holds the whole vector x, lower bound 0; its two entries are named x and y
x = pv.addMVar(2, lb=0.0, name=["x", "y"])

# objective -- mu' x   (for a 1-D array .T changes nothing; it is there so the line reads like the math)
pv.setObjective(mu.T @ x, GRB.MAXIMIZE)

# constraints -- x' Sigma x <= 0.13^2 and e' x <= 1
pv.addConstr(x.T @ Sigma @ x <= 0.13 ** 2, name="risk_cap")
pv.addConstr(e.T @ x <= 1, name="budget")

# read it back -- compare with the by-hand printout above: the same model, line for line
pv.update()
pv.write("max_return_vector.lp")
print(GREEN + open("max_return_vector.lp").read() + RESET)

pv.optimize()
print(GREEN + f"\nx = {x.X.round(4)}   expected return = {pv.ObjVal * 100:.2f}%" + RESET)

In [ ]:
# Now that we are done with writing the optimization model on the computer, let's solve the problem
pf.optimize()

def variance(x, y):
    """portfolio variance x' Sigma x -- the left-hand side of risk_cap"""
    return 0.01 * x * x + 0.04 * y * y + 0.002 * x * y

x, y = pf.getVars()          # the vector-form cell above reused the name x -- take pf's two variables back
x_opt, y_opt, c_opt = x.X, y.X, pf.ObjVal
sigma = variance(x_opt, y_opt) ** 0.5
print(GREEN + f"\nx={x_opt:.4f}  y={y_opt:.4f}")
print(f"expected return = {c_opt * 100:.2f}%   risk sigma = {sigma * 100:.1f}%" + RESET)

## Draw it on the $(x, y)$ plane

- **shaded** — the feasible region: the triangle $x + y \le 1$, $x, y \ge 0$, cut by the risk **ellipse** (dashed)
- **black line** — a level line of the objective, $0.10\,x + 0.15\,y = c$

Raise $c$ and the line slides up and to the right. The largest $c$ that still touches the shaded region is the optimum — it touches at the **corner** where the budget line meets the ellipse, so both constraints bind.

Without the cap, the line would slide on to the top vertex $(0, 1)$ — **outside** the ellipse.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X, Y = np.meshgrid(np.linspace(-0.1, 1.15, 800), np.linspace(-0.1, 1.15, 800))
feasible = (X >= 0) & (Y >= 0) & (X + Y <= 1) & (variance(X, Y) <= sigma_max ** 2)

def draw_map(c=0.13):
    plt.figure(figsize=(6.5, 6.5))
    plt.contourf(X, Y, feasible.astype(float), levels=[0.5, 1.5], colors=["#e8b4b4"])
    plt.contour(X, Y, variance(X, Y), levels=[sigma_max ** 2], colors="#c0504d", linestyles="--")
    plt.plot([0, 1], [1, 0], color="gray", lw=1)                         # budget line x + y = 1

    xs = np.array([-0.1, 1.15])
    plt.plot(xs, (c - 0.10 * xs) / 0.15, color="black", lw=2.5,         # level line 0.10x + 0.15y = c
             label=f"$0.10x + 0.15y = {c:.3f}$")

    plt.plot(x_opt, y_opt, marker="*", ms=20, color="black")
    plt.annotate(f"optimal solution\n({x_opt:.4f}, {y_opt:.4f})", (x_opt, y_opt),
                 textcoords="offset points", xytext=(12, 8))
    plt.plot(0, 1, marker="x", ms=10, mew=2, color="gray")
    plt.annotate(f"no risk cap: $\\sigma$ = {variance(0, 1) ** 0.5 * 100:.0f}%", (0, 1),
                 textcoords="offset points", xytext=(10, 2), color="gray")

    plt.axhline(0, color="black", lw=0.8)
    plt.axvline(0, color="black", lw=0.8)
    plt.xlim(-0.1, 1.15)
    plt.ylim(-0.1, 1.15)
    plt.gca().set_aspect("equal")
    plt.grid(alpha=0.3)
    plt.xlabel("$x$  (share in asset 1)")
    plt.ylabel("$y$  (share in asset 2)")
    plt.legend(loc="upper right")
    plt.show()

draw_map(c_opt)

### Slide $c$ yourself

Same picture with a slider for $c$ — push the line up until it last touches the shaded region.

In [ ]:
from ipywidgets import interact, FloatSlider

interact(draw_map, c=FloatSlider(value=0.13, min=0.0, max=0.20, step=0.005, readout_format=".3f"));